# Obtaining data
The full list of urls (status 200 only) was obtained from the Internet Archive.
Date: 11 July 2025.
The list of these URLs (rijksmuseum_archive_urls_full.csv) has been published on Zenodo: DOI: 10.5281/zenodo.21134681

In [ ]:
import requests
import pandas as pd
import os
import time

# Output path
output_folder = r'./data'
output_filename = 'rijksmuseum_archive_urls_full.csv'
output_path = os.path.join(output_folder, output_filename)
os.makedirs(output_folder, exist_ok=True)

# CDX API URL
base_url = 'http://web.archive.org/cdx/search/cdx'

# Base parameters (no limit here!)
params = {
    'url': 'www.rijksmuseum.nl/*',
    'output': 'json',
    'collapse': 'urlkey',
    'showResumeKey': 'true',
    'filter': 'statuscode:200'  # <-- Keep this filter
}

all_records = []
resume_key = None
page = 1

print("Starting download of all archived URLs...")

while True:
    if resume_key:
        params['resumeKey'] = resume_key
    else:
        params.pop('resumeKey', None)

    try:
        response = requests.get(base_url, params=params, stream=False, timeout=60)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"⚠️ Request failed on page {page}: {e}")
        print("⏳ Waiting 5 seconds before retry...")
        time.sleep(5)
        continue  # Retry the same page

    try:
        data = response.json()
    except ValueError:
        print(f"❌ Failed to parse JSON on page {page}")
        break

    if page == 1:
        columns = data[0]
        records = data[1:-1]
    else:
        records = data[:-1]

    all_records.extend(records)

    last_entry = data[-1]
    if isinstance(last_entry, list):  # No more pages
        print("✅ Download complete.")
        break

    resume_key = last_entry
    print(f"📄 Page {page} processed, total records so far: {len(all_records)}")
    page += 1

    time.sleep(1)

# Convert to DataFrame
df = pd.DataFrame(all_records, columns=columns)
df['wayback_url'] = 'https://web.archive.org/web/' + df['timestamp'] + '/' + df['original']

# Save to CSV
df.to_csv(output_path, index=False, encoding='utf-8')
print(f"\n✅ All data saved to: {output_path}")
